# Anima × ComfyUI inference diagnostics lab

학습 없이 Anima inference에서 **DiT block별 representation 통계, effective rank, spatial RMS heatmap, dtype/device**를 기록한다.

- ComfyUI와 Anima 모델 파일은 Google Drive가 아니라 **Colab 런타임에 직접 설치/다운로드**
- ComfyUI는 **Cloudflared URL**로 접속
- custom node: `Anima Inference Diagnostics`
- 결과: `/content/anima_diagnostics/<session>/`
- 마지막 셀에서 `@param`으로 지정한 Google Drive 경로로 이동

> 현재 heatmap은 Cosmos Predict2 `attn1_patch/attn2_patch`의 **Q/K/V projection 전 representation RMS map**이다. DAAM의 실제 token attention probability는 아니다.

In [ ]:
#@title 1. GPU 확인 + ComfyUI 설치 + diagnostic node 설치
import os, pathlib
!nvidia-smi

COMFY_DIR="/content/ComfyUI"
if not os.path.exists(COMFY_DIR):
    !git clone --depth 1 https://github.com/Comfy-Org/ComfyUI.git {COMFY_DIR}

%cd {COMFY_DIR}
!pip -q install -r requirements.txt
!pip -q install -U huggingface_hub pandas matplotlib pillow

node_dir=pathlib.Path(COMFY_DIR)/"custom_nodes/anima_inference_diagnostics_v2"
node_dir.mkdir(parents=True,exist_ok=True)
!wget -q https://raw.githubusercontent.com/HisameOgasahara/deep-learning-diagnostics-and-improvement/main/Temp/v2/anima_inference_diagnostics_node_v2.py -O {node_dir}/__init__.py
print("custom node:",node_dir/"__init__.py")

Wed Aug 26 07:51:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
#@title 2. Anima Base + Qwen text encoder + VAE 다운로드
from huggingface_hub import hf_hub_download
from pathlib import Path
from PIL import Image
import os,json

repo="circlestone-labs/Anima"
comfy=Path("/content/ComfyUI")
files={
 "split_files/diffusion_models/anima-base-v1.0.safetensors":comfy/"models/diffusion_models/anima-base-v1.0.safetensors",
 "split_files/text_encoders/qwen_3_06b_base.safetensors":comfy/"models/text_encoders/qwen_3_06b_base.safetensors",
 "split_files/vae/qwen_image_vae.safetensors":comfy/"models/vae/qwen_image_vae.safetensors",
}
for remote,dst in files.items():
    dst.parent.mkdir(parents=True,exist_ok=True)
    if not dst.exists():
        cached=hf_hub_download(repo_id=repo,filename=remote)
        os.symlink(cached,dst)
    print(dst,f"{dst.stat().st_size/1024**3:.2f} GiB")

# 공식 model-card workflow PNG의 embedded workflow도 있으면 등록
png=hf_hub_download(repo_id=repo,filename="example.png")
img=Image.open(png)
wf=img.info.get("workflow")
if wf:
    wf=json.loads(wf) if isinstance(wf,str) else wf
    d=comfy/"user/default/workflows"; d.mkdir(parents=True,exist_ok=True)
    (d/"Anima_official_from_model_card.json").write_text(json.dumps(wf,ensure_ascii=False,indent=2),encoding="utf-8")
    print("workflow saved")

split_files/diffusion_models/anima-base-(…): reconstructing file:   0%|          |  0.00B / 4.18GB            

split_files/diffusion_models/anima-base-(…): downloading bytes:           |  0.00B            

/content/ComfyUI/models/diffusion_models/anima-base-v1.0.safetensors 3.89 GiB


split_files/text_encoders/qwen_3_06b_bas(…): reconstructing file:   0%|          |  0.00B / 1.19GB            

split_files/text_encoders/qwen_3_06b_bas(…): downloading bytes:           |  0.00B            

...

split_files/vae/qwen_image_vae.safetenso(…): downloading bytes:           |  0.00B            

/content/ComfyUI/models/vae/qwen_image_vae.safetensors 0.24 GiB


example.png: reconstructing file:   0%|          |  0.00B / 1.19MB            

example.png: downloading bytes:           |  0.00B            

workflow saved


## ComfyUI에서 연결

공식 Anima workflow를 연 뒤 **sampler로 들어가는 마지막 MODEL 선 사이**에 `Anima Inference Diagnostics`를 끼운다.

기본값:
- blocks: `0,6,12,18,24,27` (Anima 2B는 28 blocks)
- scalar 기록: 매 model call
- spatial map + approximate effective rank: 5 model calls마다

처음엔 512×512 또는 768×768로 확인하는 편이 가볍다. 공식 Anima Base 권장 범위는 512²~1536², 30~50 steps, CFG 4~5다.

In [ ]:
#@title 3. 기존 프로세스 종료 + ComfyUI 실시간 로그 + Cloudflared
import os,re,time,signal,pathlib,subprocess,threading,requests

COMFY_DIR="/content/ComfyUI"
PORT=8188

# 1) 이전 ComfyUI / cloudflared 정리
for pf in ["/content/comfyui.pid","/content/cloudflared.pid"]:
    if os.path.exists(pf):
        try:
            os.kill(int(pathlib.Path(pf).read_text().strip()), signal.SIGTERM)
        except Exception:
            pass

!pkill -f "python main.py" || true
!pkill -f "cloudflared tunnel" || true
time.sleep(2)

# 2) Colab + trycloudflare에서 최신 ComfyUI의 cross-site 403 guard만 비활성화
#    Quick Tunnel은 이 Colab 런타임 동안만 쓰는 실습용 공개 URL이다.
server_py=pathlib.Path(COMFY_DIR)/"server.py"
s=server_py.read_text()
old="""        if 'Sec-Fetch-Site' in request.headers:
            sec_fetch_site = request.headers['Sec-Fetch-Site']
            if sec_fetch_site == 'cross-site':
                return web.Response(status=403)
"""
new="""        # Disabled for Colab + Cloudflared Quick Tunnel diagnostics lab
        # if 'Sec-Fetch-Site' in request.headers:
        #     sec_fetch_site = request.headers['Sec-Fetch-Site']
        #     if sec_fetch_site == 'cross-site':
        #         return web.Response(status=403)
"""
if old in s:
    server_py.write_text(s.replace(old,new))
    print("patched ComfyUI cross-site guard")
else:
    print("cross-site guard already patched or upstream changed")

# 3) cloudflared 준비
cf="/content/cloudflared"
if not os.path.exists(cf):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O {cf}
    !chmod +x {cf}

# 4) ComfyUI 실행 — stdout/stderr를 이 셀에 계속 표시
p=subprocess.Popen(
    ["python","main.py","--listen","127.0.0.1","--port",str(PORT),"--lowvram"],
    cwd=COMFY_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
pathlib.Path("/content/comfyui.pid").write_text(str(p.pid))

def stream_comfy():
    for line in iter(p.stdout.readline,""):
        if not line:
            break
        print("[ComfyUI]",line,end="",flush=True)

threading.Thread(target=stream_comfy,daemon=True).start()

# 5) localhost가 실제로 준비된 다음에만 tunnel 시작
for _ in range(120):
    if p.poll() is not None:
        raise RuntimeError(f"ComfyUI exited with code {p.returncode}")
    try:
        r=requests.get(f"http://127.0.0.1:{PORT}",timeout=1)
        if r.status_code < 500:
            print(f"\n✅ ComfyUI ready: HTTP {r.status_code}")
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError("ComfyUI did not open port 8188")

# 6) Cloudflared 실행
q=subprocess.Popen(
    [cf,"tunnel","--url",f"http://127.0.0.1:{PORT}","--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
pathlib.Path("/content/cloudflared.pid").write_text(str(q.pid))

# URL이 뜰 때까지만 cloudflared 로그를 보여주고,
# 이후에도 pipe는 계속 drain하되 출력은 숨긴다.
url_holder={"url":None}
def stream_cloudflared():
    for line in iter(q.stdout.readline,""):
        if not line:
            break
        if url_holder["url"] is None:
            m=re.search(r"https://[-a-z0-9]+\.trycloudflare\.com",line)
            if m:
                url_holder["url"]=m.group(0)
                print("\n🌐 OPEN:",url_holder["url"],"\n",flush=True)
            else:
                print("[cloudflared]",line,end="",flush=True)

threading.Thread(target=stream_cloudflared,daemon=True).start()

print("\nWaiting for Cloudflare URL...\n")

# 7) 셀을 계속 점유해서 ComfyUI 로그를 실시간 확인
try:
    while True:
        if p.poll() is not None:
            raise RuntimeError(f"ComfyUI exited: {p.returncode}")
        if q.poll() is not None:
            raise RuntimeError(f"cloudflared exited: {q.returncode}")
        time.sleep(1)
except KeyboardInterrupt:
    print("\nstopping...")
finally:
    for proc in (q,p):
        try:
            proc.terminate()
        except Exception:
            pass


^C
^C
cross-site guard already patched or upstream changed
[ComfyUI] [INFO] setup plugin alembic.autogenerate.schemas
[ComfyUI] [INFO] setup plugin alembic.autogenerate.tables
[ComfyUI] [INFO] setup plugin alembic.autogenerate.types
[ComfyUI] [INFO] setup plugin alembic.autogenerate.constraints
[ComfyUI] [INFO] setup plugin alembic.autogenerate.defaults
[ComfyUI] [INFO] setup plugin alembic.autogenerate.comments
[ComfyUI] [INFO] setup plugin alembic.autogenerate.checkconstraint_byname
[ComfyUI] [WARNING] WARNING: You need pytorch with cu130 or higher to use optimized CUDA operations.
[ComfyUI] WARNING WARNING WARNING
[ComfyUI] If you are on nvidia 20 series and above it is required that you update your pytorch to cu130 or higher.
[ComfyUI] 
[ComfyUI] [INFO] Found comfy_kitchen backend hip: {'available': False, 'disabled': False, 'unavailable_reason': 'PyTorch ROCm/HIP runtime not available', 'capabilities': []}
[ComfyUI] [INFO] Found comfy_kitchen backend cuda: {'available': True, 'dis

In [ ]:
#@title 4. 최신 diagnostic session을 CSV/그래프로 정리
!wget -q https://raw.githubusercontent.com/HisameOgasahara/deep-learning-diagnostics-and-improvement/main/Temp/v2/postprocess_anima_diagnostics_v2.py -O /content/postprocess_anima_diagnostics_v2.py
%run /content/postprocess_anima_diagnostics_v2.py

In [ ]:
#@title 5. 결과 미리보기
from pathlib import Path
from IPython.display import display,Image
root=Path("/content/anima_diagnostics")
ss=sorted([p for p in root.glob("*") if p.is_dir()],key=lambda p:p.stat().st_mtime)
if ss:
    latest=ss[-1]
    print(latest)
    for n in ["model_input_rms_by_call.png","attn1_q_rms_block_call_heatmap.png","effective_rank_by_call.png"]:
        p=latest/n
        if p.exists(): display(Image(filename=str(p)))

In [ ]:
#@title 6. @param Google Drive 경로로 결과 이동
from google.colab import drive
from pathlib import Path
import shutil,time
drive.mount("/content/drive")

drive_destination="/content/drive/MyDrive/AnimaDiagnostics" #@param {type:"string"}
move_instead_of_copy=True #@param {type:"boolean"}
also_transfer_comfyui_images=True #@param {type:"boolean"}

root=Path("/content/anima_diagnostics")
ss=sorted([p for p in root.glob("*") if p.is_dir()],key=lambda p:p.stat().st_mtime)
if not ss: raise RuntimeError("diagnostic session 없음")
src=ss[-1]; dstroot=Path(drive_destination); dstroot.mkdir(parents=True,exist_ok=True)
dst=dstroot/src.name
if dst.exists(): dst=dstroot/f"{src.name}_{int(time.time())}"
(shutil.move(str(src),str(dst)) if move_instead_of_copy else shutil.copytree(src,dst))
if also_transfer_comfyui_images:
    out=Path("/content/ComfyUI/output"); target=dst/"comfyui_output"; target.mkdir(parents=True,exist_ok=True)
    if out.exists():
        for p in out.rglob("*"):
            if p.is_file():
                t=target/p.relative_to(out); t.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(p,t)
print("saved to:",dst)

## 첫 관찰 포인트
- `model_input_rms_by_call.png`: sigma가 내려가며 latent/model input scale이 어떻게 변하는가
- `attn1_q_rms_block_call_heatmap.png`: 어느 DiT block이 어느 inference 시점에서 크게 변하는가
- `effective_rank_by_call.png`: 선택 block의 representation이 실질적으로 몇 방향을 쓰는가
- `maps/*.png`: flattened `[B,S,C]` patch-token representation을 현재 입력 종횡비로 복원한 공간 RMS map
- `dtype_device_table.csv`: image/text 경로의 dtype/device 불일치 여부

다음 단계에서 projected Q/K를 계측하면 DAAM류 text-token spatial attention map으로 확장할 수 있다.